In [1]:
"""
Smart Personal Expense Manager
------------------------------
A command-line tool to log expenses, see spending by category,
set a monthly budget, and keep everything saved between runs.

Run:  python expense_manager.py

Concepts practiced:
  * Functions: each job (add, view, summarize...) is its own function
  * Data structures: a dictionary holding a list of expense dictionaries
  * Conditionals: budget warnings, input validation
  * File handling: JSON for saving/loading, CSV for exporting

Data is stored in expenses.json, next to where you run the script.
"""

import csv
import json
import os
from datetime import date, datetime

DATA_FILE = "expenses.json"
CSV_FILE = "expenses_export.csv"
CURRENCY = "$"   # change to your own currency symbol

CATEGORIES = ["Food", "Transport", "Shopping", "Bills", "Fun", "Health", "Other"]


# ---------- Saving and loading (file persistence) ----------

def new_data():
    """The structure of a brand-new data file."""
    return {"monthly_budget": 0.0, "expenses": []}


def load_data():
    """Load saved data. Start fresh if the file is missing or damaged."""
    if not os.path.exists(DATA_FILE):
        return new_data()
    try:
        with open(DATA_FILE, "r") as file:
            data = json.load(file)
        # make sure the keys we rely on exist
        data.setdefault("monthly_budget", 0.0)
        data.setdefault("expenses", [])
        return data
    except (json.JSONDecodeError, OSError):
        print("Warning: could not read the data file. Starting fresh.")
        print(f"(Your old file was left alone; fix or delete {DATA_FILE}.)")
        return new_data()


def save_data(data):
    """Write data to disk. Writing to a temp file first means a crash
    mid-save can't leave you with a half-written, corrupted file."""
    temp_file = DATA_FILE + ".tmp"
    with open(temp_file, "w") as file:
        json.dump(data, file, indent=2)
    os.replace(temp_file, DATA_FILE)


# ---------- Input helpers ----------

def ask_amount(prompt):
    """Keep asking until the user enters a positive number."""
    while True:
        text = input(prompt).strip().replace(CURRENCY, "").replace(",", "")
        try:
            amount = float(text)
        except ValueError:
            print("Please enter a number, like 12.50")
            continue
        if amount <= 0:
            print("Amount must be greater than zero.")
            continue
        return round(amount, 2)


def ask_date():
    """Ask for a date; pressing Enter means today."""
    while True:
        text = input("Date (YYYY-MM-DD, Enter for today): ").strip()
        if text == "":
            return date.today().isoformat()
        try:
            return datetime.strptime(text, "%Y-%m-%d").date().isoformat()
        except ValueError:
            print("Use the format YYYY-MM-DD, for example 2026-03-15")


def ask_category():
    """Show a numbered list of categories; allow a custom one too."""
    print("\nCategories:")
    for number, name in enumerate(CATEGORIES, start=1):
        print(f"  {number}) {name}")
    while True:
        choice = input("Pick a number (or type your own category): ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(CATEGORIES):
            return CATEGORIES[int(choice) - 1]
        if choice and not choice.isdigit():
            return choice.title()
        print("Please choose a valid number or type a name.")


def ask_month():
    """Ask for a month as YYYY-MM; Enter means the current month."""
    current = date.today().strftime("%Y-%m")
    while True:
        text = input(f"Month (YYYY-MM, Enter for {current}): ").strip()
        if text == "":
            return current
        try:
            datetime.strptime(text, "%Y-%m")
            return text
        except ValueError:
            print("Use the format YYYY-MM, for example 2026-03")


def money(amount):
    return f"{CURRENCY}{amount:,.2f}"


# ---------- Calculations ----------

def expenses_in_month(data, month):
    """Return the expenses whose date starts with 'YYYY-MM'."""
    return [e for e in data["expenses"] if e["date"].startswith(month)]


def total_of(expenses):
    return sum(e["amount"] for e in expenses)


def totals_by_category(expenses):
    """Build a dictionary like {'Food': 120.5, 'Bills': 300.0}."""
    totals = {}
    for e in expenses:
        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]
    return totals


# ---------- Menu actions ----------

def add_expense(data):
    print("\n--- Add Expense ---")
    amount = ask_amount("Amount: ")
    category = ask_category()
    note = input("Note (optional): ").strip()
    when = ask_date()

    data["expenses"].append(
        {"date": when, "amount": amount, "category": category, "note": note}
    )
    save_data(data)
    print(f"Saved: {money(amount)} on {category} ({when})")

    # Budget warning for the month this expense belongs to
    budget = data["monthly_budget"]
    month = when[:7]
    if budget > 0 and month == date.today().strftime("%Y-%m"):
        spent = total_of(expenses_in_month(data, month))
        if spent > budget:
            print(f"!! Over budget by {money(spent - budget)} this month!")
        elif spent >= 0.8 * budget:
            print(f"Heads up: you've used {spent / budget:.0%} of your budget.")


def view_expenses(data):
    print("\n--- View Expenses ---")
    month = ask_month()
    items = sorted(expenses_in_month(data, month), key=lambda e: e["date"])
    if not items:
        print(f"No expenses recorded for {month}.")
        return

    print(f"\n{'#':>3}  {'Date':<10}  {'Category':<10}  {'Amount':>10}  Note")
    print("-" * 56)
    for number, e in enumerate(items, start=1):
        print(f"{number:>3}  {e['date']:<10}  {e['category']:<10}  "
              f"{money(e['amount']):>10}  {e['note']}")
    print("-" * 56)
    print(f"Total: {money(total_of(items))}")


def view_category_summary(data):
    print("\n--- Spending by Category ---")
    month = ask_month()
    items = expenses_in_month(data, month)
    if not items:
        print(f"No expenses recorded for {month}.")
        return

    totals = totals_by_category(items)
    grand_total = total_of(items)
    # sort categories from highest to lowest spending
    for category, amount in sorted(totals.items(), key=lambda pair: pair[1], reverse=True):
        share = amount / grand_total
        bar = "#" * round(share * 30)
        print(f"{category:<10} {money(amount):>10}  {share:>4.0%}  {bar}")
    print(f"{'TOTAL':<10} {money(grand_total):>10}")


def set_budget(data):
    print("\n--- Set Monthly Budget ---")
    current = data["monthly_budget"]
    if current > 0:
        print(f"Current budget: {money(current)}")
    data["monthly_budget"] = ask_amount("New monthly budget: ")
    save_data(data)
    print(f"Budget set to {money(data['monthly_budget'])}")


def view_budget_status(data):
    print("\n--- Budget Status ---")
    budget = data["monthly_budget"]
    if budget <= 0:
        print("No budget set yet. Choose 'Set monthly budget' first.")
        return

    month = date.today().strftime("%Y-%m")
    spent = total_of(expenses_in_month(data, month))
    remaining = budget - spent
    percent = spent / budget

    filled = min(20, round(percent * 20))
    bar = "[" + "#" * filled + "-" * (20 - filled) + "]"

    print(f"Month:     {month}")
    print(f"Budget:    {money(budget)}")
    print(f"Spent:     {money(spent)}  {bar} {percent:.0%}")
    if remaining >= 0:
        print(f"Remaining: {money(remaining)}")
        if percent >= 0.8:
            print("Careful, you're close to your limit.")
    else:
        print(f"Over budget by {money(-remaining)}!")


def delete_expense(data):
    print("\n--- Delete Expense ---")
    month = ask_month()
    items = sorted(expenses_in_month(data, month), key=lambda e: e["date"])
    if not items:
        print(f"No expenses recorded for {month}.")
        return

    for number, e in enumerate(items, start=1):
        print(f"{number:>3}) {e['date']}  {e['category']:<10} {money(e['amount']):>10}  {e['note']}")

    choice = input("Number to delete (Enter to cancel): ").strip()
    if not choice.isdigit() or not 1 <= int(choice) <= len(items):
        print("Cancelled.")
        return

    target = items[int(choice) - 1]
    if input(f"Delete {money(target['amount'])} on {target['category']}? (y/n): ").strip().lower() == "y":
        data["expenses"].remove(target)
        save_data(data)
        print("Deleted.")
    else:
        print("Kept.")


def export_csv(data):
    """Write all expenses to a CSV file that opens in Excel or Sheets."""
    if not data["expenses"]:
        print("Nothing to export yet.")
        return
    with open(CSV_FILE, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["date", "amount", "category", "note"])
        writer.writeheader()
        writer.writerows(sorted(data["expenses"], key=lambda e: e["date"]))
    print(f"Exported {len(data['expenses'])} expenses to {CSV_FILE}")


# ---------- Main loop ----------

MENU = """
===== Expense Manager =====
1) Add expense
2) View expenses
3) Spending by category
4) Set monthly budget
5) Budget status
6) Delete an expense
7) Export to CSV
8) Quit"""


def main():
    data = load_data()
    actions = {
        "1": add_expense,
        "2": view_expenses,
        "3": view_category_summary,
        "4": set_budget,
        "5": view_budget_status,
        "6": delete_expense,
        "7": export_csv,
    }

    while True:
        print(MENU)
        choice = input("Choose an option: ").strip()
        if choice == "8":
            print("Your data is saved. Goodbye!")
            break
        action = actions.get(choice)
        if action:
            action(data)
        else:
            print("Please enter a number from 1 to 8.")


if __name__ == "__main__":
    main()


===== Expense Manager =====
1) Add expense
2) View expenses
3) Spending by category
4) Set monthly budget
5) Budget status
6) Delete an expense
7) Export to CSV
8) Quit


Choose an option:  1



--- Add Expense ---


Amount:  2546311



Categories:
  1) Food
  2) Transport
  3) Shopping
  4) Bills
  5) Fun
  6) Health
  7) Other


Pick a number (or type your own category):  1,2,3,5
Note (optional):  
Date (YYYY-MM-DD, Enter for today):  2026-10-22


Saved: $2,546,311.00 on 1,2,3,5 (2026-10-22)

===== Expense Manager =====
1) Add expense
2) View expenses
3) Spending by category
4) Set monthly budget
5) Budget status
6) Delete an expense
7) Export to CSV
8) Quit


Choose an option:  8


Your data is saved. Goodbye!
